In [3]:
#import requests
from selenium import webdriver
import time
import pandas as pd
import random

In [289]:
search_query = 'https://comprar.gob.ar/Item/Item/BuscarItemCiudadano'
driver = webdriver.Chrome(executable_path='C:/chromedriver/chromedriver.exe')

In [290]:
items_details = []
items_info=[]

In [291]:
driver.get(search_query)
time.sleep(random.uniform(3.0,5.0))


In [ ]:
#realiza la extracción de datos desde la tabla de resultados de Items
#Localiza todas las filas (tr) dentro de la tabla, itera sobre ellas y, para cada fila, obtiene las celdas (td) correspondientes a distintos atributos del ítem (código, rubro, clase, descripción y estado). 
#Cada valor se captura con manejo de errores para evitar que el scraping se interrumpa si falta algún dato. 
#Finalmente, construye una lista con esos campos y la agrega a la estructura global items_details, acumulando todos los registros extraídos.

In [256]:
def obtiene_datos_pag(): #agregar un try catch por la cantidad de elementos menor a 10
    print('cuenta el nro de rows')
    rows = driver.find_elements_by_xpath('//*[@id="divResultados"]/table/tbody/tr')#obtiene el nro de rows en la tabla
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla : '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
            #note: index start from 0, 1 is col 2
        if number_of_cols > 0:
            try:
                codigo_del_item=cols[0].text # nro
            except Exception:
                codigo_del_item=''
                print('error codigo')
            try:
                rubro = cols[1].text
            except Exception:
                rubro=''
                print('error rubro')
            try:
                clase = cols[2].text
            except Exception:
                clase=''
                print('error clase')
            try:
                item = cols[3].text.replace(';',' -')
                #print(item)
            except Exception:
                item=''
                print('error item')
            try:
                estado=cols[4].text
            except Exception:
                estado=''
                print('error estado')
         #   print(estado)
        items_info = [codigo_del_item,rubro,clase,item,estado]
        items_details.append(items_info) 
    


In [193]:
obtiene_datos_pag()

cuenta el nro de rows
nro de rows a iterar dentro de la tabla : 7
GAS BUTANO - USO: ENCENDEDORES, PRESENTACION: 770 CM3
GAS BUTANO - USO: SIN VALOR, PRESENTACION: CARTUCHO X 227 GR
GAS BUTANO - USO: ENCENDEDORES, PRESENTACION: ENVASE X 180 CM3
GAS BUTANO - USO: SOLDADURA, PRESENTACION: A GRANEL
GAS BUTANO - USO: ENCENDEDOR, PRESENTACION: ENVASE X 440 CM3
GAS BUTANO - USO: SOLDADURA, PRESENTACION: ENVASE X 200 mL
GAS BUTANO - USO: FAROL, PRESENTACION: ENVASE X 190 GR


In [ ]:
#Esta función se encarga de la persistencia de los datos recolectados, 
#transformando la lista items_details en un DataFrame de pandas con columnas definidas (código, rubro, clase, ítem y estado) 
#y exportándolo a un archivo CSV. El archivo se guarda con separador | y codificación UTF-8 con BOM para compatibilidad con Excel. 


In [77]:
#guarda datos pagina
#grabar cada linea en un csv
def guarda_datos_items():
    #print(contratos_details)
    datos=pd.DataFrame(items_details, columns =['codigo_del_item', 'rubro','clase','item','estado'])
    #print(datos)
    datos.to_csv(path_or_buf='C:/Users/Vanina/Documents/tesis/Datos/procesos/items_detalle.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)


In [80]:
guarda_datos_items()

In [292]:
#navega por las paginas

items_details = []
items_info=[]

#primero obtener la información de la primer página y luego recorrer las siguientes:
try:
    obtiene_datos_pag()
    print('sale')
    guarda_datos_items()
except Exception:
    print('Error en la primer página')


#obtiene texto para ver el nro de casos
texto_nro_casos = driver.find_element_by_xpath('//*[@id="divResultados"]/h4/small/span')
print(texto_nro_casos.text)
indice1 = texto_nro_casos.text.find("encontrado ")
print(indice1)
indice2=texto_nro_casos.text.find("resultados")
print(indice2)
print(texto_nro_casos.text[indice1+11:indice2])
#como cada página tiene 10 lineas se obtiene el nro de páginas a iterar
f_nro_casos=float(texto_nro_casos.text[indice1+11:indice2])/10
print(f_nro_casos)
nro_casos, d = divmod(f_nro_casos, 1)
if d > 0:
    nro_casos = nro_casos + 1
    
nro_casos = int(nro_casos)
print('nro de casos: '+str(nro_casos))
i=1 #hojas totales
indice=2 # por tabla max 11

while i < nro_casos+1:
    i = i +1
    try:
        link_pagina = driver.find_element_by_css_selector('#divResultados > div > ul > li.PagedList-skipToNext > a')
        #link_pagina = driver.find_element_by_css_selector('#divResultados > div > ul > li:nth-child('+str(indice)+') > a')
        #ctl00_CPH1_GridListaPliegos > tbody > tr.pagination-gv > td > table > tbody > tr > td:nth-child('+str(indice)+') > a')
        link_pagina.click() 
        time.sleep(random.uniform(1,1.5)) 
        obtiene_datos_pag()
            
    except Exception:
        print('Error en la página: '+str(i))
    if (i == 2):
        print('entra 1')
        indice = 4
    else:
        if(i >= 3)and (i < 8):
            print('entra 2')
            indice = indice + 1
        else:
            indice = 10
    print('indice: '+str(indice))
    print('i: '+str(i))
    print('------------------------------------------------------------------------------')

guarda_datos_items()   

cuenta el nro de rows
nro de rows a iterar dentro de la tabla : 10
sale
Se han encontrado 112 resultados para su búsqueda
7
22
112 
11.2
nro de casos: 12
cuenta el nro de rows
nro de rows a iterar dentro de la tabla : 10
entra 1
indice: 4
i: 2
------------------------------------------------------------------------------
cuenta el nro de rows
nro de rows a iterar dentro de la tabla : 10
entra 2
indice: 5
i: 3
------------------------------------------------------------------------------
cuenta el nro de rows
nro de rows a iterar dentro de la tabla : 10
entra 2
indice: 6
i: 4
------------------------------------------------------------------------------
cuenta el nro de rows
nro de rows a iterar dentro de la tabla : 10
entra 2
indice: 7
i: 5
------------------------------------------------------------------------------
cuenta el nro de rows
nro de rows a iterar dentro de la tabla : 10
entra 2
indice: 8
i: 6
------------------------------------------------------------------------------
c